# synthgen — walkthrough

A guided tour of the library in `src/synthgen/`.

```bash
pip install -e ".[pandas]"
cp .env.example .env    # then paste your key
```


## 1. The schema is the contract

In [ ]:
from synthgen import Person
from synthgen.schemas import field_summary

print(field_summary(Person))

In [ ]:
# Loose model dates are coerced; impossible ones are rejected
from pydantic import ValidationError

base = dict(full_name="Ada Lovelace", email="ada@example.com",
            street_address="12 Analytical Way", city="London",
            country="United Kingdom", occupation="Mathematician")

print(Person(**base, date_of_birth="04/05/1990").date_of_birth)
try:
    Person(**base, date_of_birth="the nineties")
except ValidationError as exc:
    print("rejected:", exc.errors()[0]["msg"])

## 2. Generate

Needs a real key from here on.

In [ ]:
from openai import OpenAI

from synthgen import Settings, SyntheticDataGenerator

settings = Settings.from_env()
gen = SyntheticDataGenerator(
    OpenAI(api_key=settings.api_key),
    Person,
    model=settings.model,
    batch_size=20,
    concurrency=4,
    seed=42,
)
result = gen.generate(60, extra_instructions="Mix of European and South Asian backgrounds.")
result.stats.as_dict()

In [ ]:
df = result.to_dataframe()
df.head()

## 3. Check it before you trust it

Generated data fails quietly — 500 rows that are really 40 repeated. Check before you trust.

In [ ]:
from synthgen.quality import report

print(report(result.to_dicts()).render())

## 4. Export

In [ ]:
from synthgen.exporters import export

export(result.to_dicts(), "out/people.csv")
export(result.to_dicts(), "out/people.jsonl")

## 5. A custom schema, no Python

In [ ]:
from synthgen import schema_from_spec

Product = schema_from_spec("../examples/product_schema.json")
product_gen = SyntheticDataGenerator(OpenAI(api_key=settings.api_key), Product, batch_size=15)
products = product_gen.generate(30)
products.to_dataframe().head()

## 6. Testing without spending anything

In [ ]:
import sys

sys.path.insert(0, "../tests")

from conftest import FakeClient, unique_person_handler  # noqa: E402

offline = SyntheticDataGenerator(FakeClient(handler=unique_person_handler), Person, batch_size=10)
print(offline.generate(25).stats.as_dict())